# Indian Legal LLM - Complete Setup (Colab)

This notebook runs the **entire Legal LLM project** on Google Colab:
1. Uploads model artifacts from Google Drive
2. Installs all dependencies
3. Starts the FastAPI backend
4. Builds & serves the React frontend
5. Creates a public URL via ngrok

**Requirements:** Set runtime to **GPU → T4** (Runtime → Change runtime type → T4 GPU)

## Step 0: Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime → Change runtime type → T4 GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu} | VRAM: {vram:.1f} GB")

## Step 1: Mount Google Drive & Copy Project

Upload the entire `Leagal LLM` folder to your Google Drive first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os

# ─── CHANGE THIS PATH if your folder is in a different Drive location ───
DRIVE_PROJECT = "/content/drive/MyDrive/Leagal LLM"
LOCAL_PROJECT = "/content/LegalLLM"

if not os.path.exists(DRIVE_PROJECT):
    raise FileNotFoundError(
        f"Project not found at: {DRIVE_PROJECT}\n"
        "Upload the 'Leagal LLM' folder to your Google Drive root, or edit DRIVE_PROJECT above."
    )

# Copy to local disk for faster I/O
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
print(f"Copied project to {LOCAL_PROJECT}")
os.listdir(LOCAL_PROJECT)

## Step 2: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn pydantic torch transformers accelerate \
    peft sentencepiece protobuf bitsandbytes pyngrok

## Step 3: Verify Model Files Exist

In [ ]:
import os

model_dir = "/content/LegalLLM/backend/model_artifacts/phi3-legal-lora"
assert os.path.exists(model_dir), f"Model dir not found: {model_dir}"

files = os.listdir(model_dir)
print(f"Model directory has {len(files)} files:")
for f in sorted(files):
    size = os.path.getsize(os.path.join(model_dir, f)) if os.path.isfile(os.path.join(model_dir, f)) else 0
    print(f"  {f:40s}  {size/1024/1024:.1f} MB" if size else f"  {f}/")

## Step 4: Quick Test — Load Model & Generate

In [ ]:
import sys, time
sys.path.insert(0, "/content/LegalLLM/backend")

from LLM.llm import load_model, generate_response

print("Loading model...")
t0 = time.time()
model, tokenizer = load_model()
print(f"Model loaded in {time.time()-t0:.1f}s")

In [ ]:
# Quick inference test
test_queries = [
    "What is Section 302 of the Indian Penal Code?",
    "What are the fundamental rights under the Indian Constitution?",
    "Explain bail provisions under CrPC",
]

for q in test_queries:
    print(f"\nQ: {q}")
    print("-" * 60)
    t0 = time.time()
    ans = generate_response(q, max_tokens=256)
    print(f"A ({time.time()-t0:.1f}s): {ans}")
    print("=" * 60)

## Step 5: Start FastAPI Backend + ngrok Public URL

In [ ]:
# Set your ngrok auth token (free at https://ngrok.com)
# You only need to do this once
NGROK_TOKEN = ""  # <-- paste your ngrok token here

if NGROK_TOKEN:
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = NGROK_TOKEN
    print("ngrok token set!")
else:
    print("⚠️  No ngrok token set. Paste your free token from https://ngrok.com")
    print("   Without it, the public URL won't work (local API still works).")

In [ ]:
import subprocess, threading, time

# Start uvicorn in background
server_proc = subprocess.Popen(
    ["uvicorn", "Api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/LegalLLM/backend",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Print server logs in background
def stream_logs():
    for line in iter(server_proc.stdout.readline, b""):
        print(f"[server] {line.decode().strip()}")

threading.Thread(target=stream_logs, daemon=True).start()
time.sleep(3)
print("FastAPI server started on port 8000")

In [ ]:
# Health check
import requests
resp = requests.get("http://127.0.0.1:8000/api/health")
print(resp.json())

In [ ]:
# Create public URL with ngrok
try:
    from pyngrok import ngrok
    public_url = ngrok.connect(8000, "http").public_url
    print(f"\n{'='*60}")
    print(f"PUBLIC URL: {public_url}")
    print(f"API Health: {public_url}/api/health")
    print(f"API Docs:   {public_url}/docs")
    print(f"Frontend:   {public_url}/")
    print(f"{'='*60}\n")
except Exception as e:
    print(f"ngrok failed: {e}")
    print("You can still use the API at http://127.0.0.1:8000")

## Step 6: Test API via Public URL

In [ ]:
import requests, json

# Use local URL (works even without ngrok)
BASE = "http://127.0.0.1:8000"

queries = [
    "What is Section 302 of IPC?",
    "What are the fundamental rights under the Indian Constitution?",
    "What is the process of filing a bail application in India?",
]

for q in queries:
    print(f"\nQ: {q}")
    resp = requests.post(f"{BASE}/api/inference", json={"query": q, "max_tokens": 512})
    data = resp.json()
    time_sec = data.get("time_sec", "N/A")
    print(f"A ({time_sec}s): {data.get('response', data)}")
    print("-" * 60)

## Step 7: Inline Frontend (works inside Colab!)

If you don't want to set up React/npm, this embedded HTML frontend works directly in the notebook.

In [ ]:
from IPython.display import display, HTML

# Use the ngrok public URL if available, else localhost
try:
    api_url = public_url
except:
    api_url = "http://127.0.0.1:8000"

html_ui = f"""
<style>
.legal-app *{{margin:0;padding:0;box-sizing:border-box}}
.legal-app{{font-family:Inter,system-ui,sans-serif;max-width:860px;margin:0 auto;border-radius:20px;overflow:hidden;box-shadow:0 8px 40px rgba(0,0,0,.4);border:1px solid #27272a}}
.legal-header{{background:linear-gradient(135deg,#7c3aed,#4f46e5);color:#fff;padding:24px 28px;display:flex;align-items:center;gap:14px}}
.legal-header .icon{{width:48px;height:48px;border-radius:14px;background:rgba(255,255,255,.15);display:flex;align-items:center;justify-content:center}}
.legal-header h2{{font-size:20px;font-weight:700;margin:0}}
.legal-header p{{font-size:13px;margin:4px 0 0;opacity:.8}}
.legal-header .status{{margin-left:auto;display:flex;align-items:center;gap:6px;font-size:11px;opacity:.85}}
.legal-header .status-dot{{width:8px;height:8px;border-radius:50%;background:#ef4444;animation:lPulse 1.5s infinite}}
.legal-header .status-dot.online{{background:#22c55e}}
.legal-cards{{display:grid;grid-template-columns:repeat(3,1fr);gap:10px;padding:16px 20px;background:#18181b;border-bottom:1px solid #27272a}}
.legal-card{{background:#27272a;border:1px solid #3f3f46;border-radius:10px;padding:12px;cursor:pointer;transition:all .2s;text-align:left}}
.legal-card:hover{{border-color:#7c3aed;background:#1e1b2e;transform:translateY(-1px)}}
.legal-card strong{{display:block;font-size:12px;color:#c084fc;margin-bottom:3px}}
.legal-card span{{font-size:11px;color:#71717a}}
#legalChatBox{{background:#0f0f13;padding:20px;height:420px;overflow-y:auto;color:#e4e4e7;font-size:14px;line-height:1.6;scroll-behavior:smooth}}
#legalChatBox::-webkit-scrollbar{{width:5px}}
#legalChatBox::-webkit-scrollbar-thumb{{background:#3f3f46;border-radius:5px}}
.legal-msg{{display:flex;gap:10px;margin-bottom:16px;animation:lSlide .3s ease}}
.legal-msg.user{{justify-content:flex-end}}
.legal-avatar{{width:32px;height:32px;border-radius:10px;display:flex;align-items:center;justify-content:center;flex-shrink:0;font-size:12px;font-weight:700}}
.legal-msg.user .legal-avatar{{background:linear-gradient(135deg,#7c3aed,#a855f7);color:#fff}}
.legal-msg.bot .legal-avatar{{background:#27272a;color:#c084fc;border:1px solid #3f3f46}}
.legal-bubble-wrap{{max-width:70%;display:flex;flex-direction:column}}
.legal-bubble{{padding:10px 14px;border-radius:12px;white-space:pre-wrap;word-break:break-word;line-height:1.6}}
.legal-msg.user .legal-bubble{{background:linear-gradient(135deg,#7c3aed,#6d28d9);color:#fff;border-bottom-right-radius:4px}}
.legal-msg.bot .legal-bubble{{background:#1e1e24;color:#d4d4d8;border:1px solid #27272a;border-bottom-left-radius:4px}}
.legal-meta{{display:flex;align-items:center;gap:6px;margin-top:3px;padding:0 4px}}
.legal-time{{font-size:10px;color:#52525b}}
.legal-copy{{background:none;border:none;color:#52525b;font-size:10px;cursor:pointer;padding:2px 5px;border-radius:3px;display:flex;align-items:center;gap:3px}}
.legal-copy:hover{{color:#a1a1aa;background:#27272a}}
.legal-copy.copied{{color:#22c55e}}
.typing-anim{{display:flex;gap:4px;padding:4px 0}}
.typing-anim span{{width:6px;height:6px;background:#71717a;border-radius:50%;animation:lBlink 1.4s infinite both}}
.typing-anim span:nth-child(2){{animation-delay:.2s}}
.typing-anim span:nth-child(3){{animation-delay:.4s}}
.legal-input-area{{display:flex;gap:8px;padding:14px 20px;background:#18181b;border-top:1px solid #27272a;align-items:flex-end}}
.legal-input-area textarea{{flex:1;padding:10px 14px;border-radius:10px;border:1px solid #3f3f46;background:#27272a;color:#e4e4e7;font-size:14px;font-family:inherit;outline:none;resize:none;max-height:100px;line-height:1.4;transition:border-color .2s}}
.legal-input-area textarea:focus{{border-color:#7c3aed;box-shadow:0 0 0 2px rgba(124,58,237,.1)}}
.legal-input-area textarea::placeholder{{color:#52525b}}
.legal-send{{width:40px;height:40px;border-radius:10px;border:none;background:#7c3aed;color:#fff;cursor:pointer;display:flex;align-items:center;justify-content:center;transition:all .2s;flex-shrink:0}}
.legal-send:hover{{background:#6d28d9;transform:scale(1.05)}}
.legal-send:disabled{{background:#3f3f46;cursor:not-allowed;transform:none}}
.legal-disclaimer{{text-align:center;font-size:10px;color:#52525b;padding:8px 20px 12px;background:#18181b}}
@keyframes lSlide{{from{{opacity:0;transform:translateY(8px)}}to{{opacity:1;transform:translateY(0)}}}}
@keyframes lBlink{{0%,80%,100%{{opacity:.3}}40%{{opacity:1}}}}
@keyframes lPulse{{0%,100%{{opacity:1}}50%{{opacity:.5}}}}
</style>

<div class="legal-app">
  <div class="legal-header">
    <div class="icon">
      <svg width="24" height="24" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="1.5">
        <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
      </svg>
    </div>
    <div>
      <h2>Indian Legal Assistant</h2>
      <p>Powered by Fine-tuned Phi-3 on Indian Law</p>
    </div>
    <div class="status">
      <div class="status-dot" id="lStatusDot"></div>
      <span id="lStatusText">Checking...</span>
    </div>
  </div>
  <div class="legal-cards">
    <div class="legal-card" onclick="lUsePrompt('What is Section 420 of IPC?')"><strong>IPC Section 420</strong><span>Cheating & dishonesty</span></div>
    <div class="legal-card" onclick="lUsePrompt('Explain Article 14 of the Indian Constitution')"><strong>Article 14</strong><span>Right to Equality</span></div>
    <div class="legal-card" onclick="lUsePrompt('What is the process of filing an FIR in India?')"><strong>Filing an FIR</strong><span>Process & requirements</span></div>
    <div class="legal-card" onclick="lUsePrompt('Explain bail provisions under CrPC')"><strong>Bail Provisions</strong><span>CrPC bail process</span></div>
    <div class="legal-card" onclick="lUsePrompt('What are the fundamental rights under the Indian Constitution?')"><strong>Fundamental Rights</strong><span>Part III of Constitution</span></div>
    <div class="legal-card" onclick="lUsePrompt('Explain the Right to Information Act 2005')"><strong>RTI Act 2005</strong><span>Right to Information</span></div>
  </div>
  <div id="legalChatBox"></div>
  <div class="legal-input-area">
    <textarea id="legalInput" rows="1" placeholder="Ask about Indian law..." onkeydown="if(event.key==='Enter'&&!event.shiftKey){{event.preventDefault();lSend()}}" oninput="this.style.height='auto';this.style.height=Math.min(this.scrollHeight,100)+'px'"></textarea>
    <button class="legal-send" id="legalSendBtn" onclick="lSend()">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2">
        <line x1="22" y1="2" x2="11" y2="13"/><polygon points="22 2 15 22 11 13 2 9 22 2"/>
      </svg>
    </button>
  </div>
  <div class="legal-disclaimer">This AI provides general legal information, not legal advice. Consult a lawyer for specific cases.</div>
</div>

<script>
const LAPI = "{api_url}";
const lBox = document.getElementById('legalChatBox');
const lInp = document.getElementById('legalInput');
const lBtn = document.getElementById('legalSendBtn');
let lLoading = false;
let lMsgIdx = 0;

/* health check */
(async function lHealthCheck() {{
  const dot = document.getElementById('lStatusDot');
  const txt = document.getElementById('lStatusText');
  try {{
    const r = await fetch(LAPI + '/api/health');
    if (r.ok) {{ dot.classList.add('online'); txt.textContent = 'Online'; }}
    else {{ txt.textContent = 'Offline'; }}
  }} catch(e) {{ txt.textContent = 'Offline'; }}
}})();

function lFmtTime() {{ return new Date().toLocaleTimeString([], {{hour:'2-digit',minute:'2-digit'}}); }}

function lAddMsg(role, text, timeSec) {{
  const idx = lMsgIdx++;
  const avSvg = role==='bot'
    ? '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/></svg>'
    : '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M20 21v-2a4 4 0 00-4-4H8a4 4 0 00-4 4v2"/><circle cx="12" cy="7" r="4"/></svg>';
  const copyBtn = role==='bot' ? `<button class="legal-copy" onclick="lCopy(${{idx}})" id="lCopy${{idx}}"><svg width="12" height="12" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg> Copy</button>` : '';
  const timeLabel = (role==='bot' && timeSec) ? `<span class="legal-time">${{timeSec}}s</span>` : '';
  const d = document.createElement('div');
  d.className = 'legal-msg ' + role;
  d.id = 'lMsg' + idx;
  d.setAttribute('data-text', text);
  d.innerHTML = (role==='bot'?`<div class="legal-avatar">${{avSvg}}</div>`:'')+
    `<div class="legal-bubble-wrap"><div class="legal-bubble">${{lEsc(text)}}</div><div class="legal-meta"><span class="legal-time">${{lFmtTime()}}</span>${{timeLabel}}${{copyBtn}}</div></div>`+
    (role==='user'?`<div class="legal-avatar">${{avSvg}}</div>`:'');
  lBox.appendChild(d);
  lBox.scrollTop = lBox.scrollHeight;
  return idx;
}}

function lEsc(s) {{ const d=document.createElement('div'); d.textContent=s; return d.innerHTML; }}

function lCopy(idx) {{
  const el = document.getElementById('lMsg'+idx);
  const text = el?.getAttribute('data-text') || '';
  navigator.clipboard.writeText(text).then(() => {{
    const btn = document.getElementById('lCopy'+idx);
    if(btn){{ btn.classList.add('copied'); btn.innerHTML='<svg width="12" height="12" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polyline points="20 6 9 17 4 12"/></svg> Copied!'; }}
    setTimeout(()=>{{ if(btn){{ btn.classList.remove('copied'); btn.innerHTML='<svg width="12" height="12" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="9" y="9" width="13" height="13" rx="2" ry="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg> Copy'; }} }}, 2000);
  }});
}}

function lShowTyping() {{
  const d = document.createElement('div');
  d.className = 'legal-msg bot'; d.id = 'lTyping';
  d.innerHTML = '<div class="legal-avatar"><svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/></svg></div><div class="legal-bubble-wrap"><div class="legal-bubble"><div class="typing-anim"><span></span><span></span><span></span></div></div></div>';
  lBox.appendChild(d); lBox.scrollTop=lBox.scrollHeight;
}}
function lRemoveTyping() {{ const t=document.getElementById('lTyping'); if(t)t.remove(); }}

async function lSend() {{
  const q = lInp.value.trim();
  if (!q || lLoading) return;
  lInp.value = ''; lInp.style.height='auto';
  lLoading = true; lBtn.disabled = true;
  lAddMsg('user', q);
  lShowTyping();
  try {{
    const r = await fetch(LAPI + '/api/inference', {{
      method: 'POST', headers: {{'Content-Type': 'application/json'}},
      body: JSON.stringify({{query: q, max_tokens: 512}})
    }});
    const data = await r.json();
    lRemoveTyping();
    lAddMsg('bot', data.response || data.detail || 'No response', data.time_sec);
  }} catch(e) {{
    lRemoveTyping();
    lAddMsg('bot', 'Error: ' + e.message);
  }}
  lLoading = false; lBtn.disabled = false;
  lInp.focus();
}}

function lUsePrompt(q) {{ lInp.value = q; lSend(); }}
</script>
"""

display(HTML(html_ui))

## (Optional) Build React Frontend

If you want the full React frontend instead of the inline one above:

In [ ]:
# Only run this if you want the full React build
# This takes ~2 minutes on Colab

# !curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash -
# !sudo apt-get install -y nodejs
# !cd /content/LegalLLM/frontend && npm install && npm run build
# print("React build complete! Served at the public URL above.")

---
## Stop Server

In [ ]:
# Run this when you're done
server_proc.terminate()
try:
    from pyngrok import ngrok
    ngrok.kill()
except:
    pass
print("Server stopped.")